# Fusión con `mergekit` (promedio simple y TIES) — Google Colab

Fusiona los tres adaptadores LoRA (`generador1/2/3`) con **mergekit**, que opera sobre modelos **completos**: primero se incorpora cada adaptador al modelo base (`merge_and_unload`), luego se fusiona con un **promedio simple** (`linear`) y con **TIES**, y se evalúan igual que el resto de modelos de la Fase 3 (mismo test común, mismo BLEU/chrF).

**Por qué importa:** la fusión "lineal" hecha con PEFT (sobre las matrices A y B del adaptador) quedó *por debajo del modelo base*. Aquí el promedio se hace sobre los pesos completos, que equivale a base + promedio de las tres actualizaciones: si ahora funciona, la causa era cómo se promediaba, no la idea de fusionar.

**Requisitos:** `mergekit` 0.1.4 es incompatible con `transformers` 5.x (falla con `PydanticUserError: ConfiguredModuleArchitecture is not fully defined`); esta libreta instala `transformers<5`. Necesita ~25 GB de disco (3 modelos de ~6 GB + 2 fusiones). No requiere GPU para fusionar, pero sí para evaluar (T4).

**Todo error queda en `merging/logs/`** (se incluye en el zip final): si algo falla, no lo corrijas a mano; pásalo tal cual, es información para el paper.

In [ ]:
!nvidia-smi

In [ ]:
import os, json, time, subprocess
%cd /content
if not os.path.isdir("/content/traductor-jerga-dialectal-slm"):
    !git clone -b develop https://github.com/Anderfg13/traductor-jerga-dialectal-slm.git
%cd /content/traductor-jerga-dialectal-slm
!git pull -q origin develop
os.makedirs("merging/logs", exist_ok=True)
TIEMPOS_PATH = "merging/logs/tiempos_mergekit.json"
def registrar(nombre, t0):
    t = json.load(open(TIEMPOS_PATH)) if os.path.exists(TIEMPOS_PATH) else {}
    t[nombre] = round(time.time() - t0, 1)
    json.dump(t, open(TIEMPOS_PATH, "w"), indent=1)
    print(f"[tiempo] {nombre}: {t[nombre] / 60:.1f} min")

In [ ]:
# Dependencias. mergekit fija versiones de pydantic/safetensors/accelerate y NO funciona con transformers 5.x.
!pip install -q mergekit "transformers<5" "huggingface_hub<1" peft sacrebleu 2>&1 | tail -3
!pip uninstall -y -q torchao
import transformers
print("transformers", transformers.__version__, "(debe ser 4.x)")
assert transformers.__version__.startswith("4"), "mergekit necesita transformers 4.x: reinicia el entorno y vuelve a correr esta celda"
!mergekit-yaml --help | head -3

## Paso 1 — Incorporar cada adaptador al modelo base (modelo completo, float16)

In [ ]:
t0 = time.time()
!python merging/incorporar_adaptadores.py 2>&1 | tee merging/logs/incorporar_adaptadores.log
registrar("incorporar_adaptadores", t0)
!du -sh merging/modelos_completos/*

## Paso 2 — Promedio simple de pesos (`linear`)

In [ ]:
t0 = time.time()
!mergekit-yaml merging/configs/linear.yaml merging/salida/linear --copy-tokenizer --lazy-unpickle $( [ -n "$(nvidia-smi -L 2>/dev/null)" ] && echo --cuda ) 2>&1 | tee merging/logs/mergekit_linear.log | tail -15
registrar("mergekit_linear", t0)

## Paso 3 — TIES-merging (`ties`, base = Qwen2.5-3B-Instruct)

In [ ]:
t0 = time.time()
!mergekit-yaml merging/configs/ties.yaml merging/salida/ties --copy-tokenizer --lazy-unpickle $( [ -n "$(nvidia-smi -L 2>/dev/null)" ] && echo --cuda ) 2>&1 | tee merging/logs/mergekit_ties.log | tail -15
registrar("mergekit_ties", t0)

## Paso 4 — Prueba de coherencia (5 frases a mano)

Cada modelo debe cargar sin errores y producir traducciones que no sean basura ni texto repetido. Se imprimen las 5 traducciones: **léelas**; las heurísticas no sustituyen leerlas.

In [ ]:
for nombre in ["linear", "ties"]:
    print("=" * 20, nombre)
    !python merging/probar_fusion.py merging/salida/{nombre} 2>&1 | tee merging/logs/probar_{nombre}.log

## Paso 5 — Evaluación con el mismo test común y las mismas métricas

In [ ]:
os.makedirs("evaluation/reportes", exist_ok=True)
for nombre in ["linear", "ties"]:
    t0 = time.time()
    !python evaluation/generar_predicciones.py --nombre mergekit_{nombre} --modelo-completo merging/salida/{nombre}
    registrar(f"predicciones_mergekit_{nombre}", t0)
    !python evaluation/metricas_automaticas.py --predicciones evaluation/predicciones/mergekit_{nombre}.json --referencias evaluation/test_comun.json --campo-prediccion traduccion_modelo --salida evaluation/reportes/mergekit_{nombre}.md --salida-json evaluation/reportes/mergekit_{nombre}.json | head -12
!python evaluation/tabla_comparativa.py

## Paso 6 — Empaquetar (solo resultados y logs; los modelos de ~6 GB NO se descargan)

In [ ]:
!rm -f resultados_mergekit.zip
!zip -rq resultados_mergekit.zip merging/logs merging/salida/linear/mergekit_config.yml merging/salida/ties/mergekit_config.yml evaluation/predicciones/mergekit_linear.json evaluation/predicciones/mergekit_ties.json evaluation/reportes/mergekit_linear.json evaluation/reportes/mergekit_linear.md evaluation/reportes/mergekit_ties.json evaluation/reportes/mergekit_ties.md evaluation/comparacion_fase3.md
!ls -lh resultados_mergekit.zip
from google.colab import files
files.download("resultados_mergekit.zip")